In [ ]:
from pathlib import Path
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["input", "output"])
args = Args("epfl1_subset1_float.mrc",
            "epfl1_subset1_float.pdf")

In [ ]:
from my_google_auth import DriveHandler
service = DriveHandler.get_drive_service()
handler = DriveHandler.DriveHandler(service)

In [ ]:
DRIVE_TOMOGRADENOISING_TMP       = '1hGHvkP46fxLCQbUlyYhAS_eVl6PollQM'  # "tmp" folder
DRIVE_TOMOGRADENOISING_TOMOGRAMS = '1hfAOv6etLjB16K-nCrg-0u24iZBvmr1-'  # "Tomograms" folder

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        file_id = handler.find_file_id(drive_file_name=output, drive_folder_id=DRIVE_TOMOGRADENOISING_TMP)
        if file_id == None:
            print(f"{output} does not exist in Google Drive. Creating ...")
        else:
            print(f"Downloading {output} from Google Drive")
            success = handler.download(file_id, local_save_path=output)

In [ ]:
from collections import namedtuple
import mrcfile
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
#import matplotlib.ticker as mticker
from matplotlib.pyplot import figure

In [ ]:
# apt install cm-super-minimal
# apt install dvipng
plt.rcParams.update({
    "text.usetex": True,
    #"font.family": "Helvetica",
    "font.family": "Serif",
    "text.latex.preamble": r"\usepackage{amsmath} \usepackage{amsfonts}"
})

In [ ]:
file_path = Path(args.input)
if file_path.exists():
    print(f"Found local {args.input}")
else:
    file_id = handler.find_file_id(drive_file_name=args.input, drive_folder_id=DRIVE_TOMOGRADENOISING_TOMOGRAMS)
    success = handler.download(file_id, local_save_path=args.input)

In [ ]:
vol_MRC = mrcfile.open(args.input)
vol = vol_MRC.data
print(f"shape={vol.shape}")

In [ ]:
figure(figsize=(16, 16))
slice_idx = vol.shape[0]//2
plt.imshow(vol[slice_idx, 300:, 600:], cmap="gray")
#plt.imshow(vol[slice_idx], cmap="gray")
plt.savefig(args.output, bbox_inches='tight')

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        uploaded_file_id = handler.upload(
            local_file_path=output,
            drive_file_name=output,
            drive_folder_id=DRIVE_TOMOGRADENOISING_TMP)